# SprayLine 感測資料計算公式

本文件整理 SprayLine 系統中各物件的計算方式，目標是說明：

1. 每個物件使用哪些**確定會有的 sensor data**。
2. 如何用目前值與過去值計算**線性變化率**。
3. 若物件變化不是線性的，可如何補充**非線性模型**。
4. 非線性公式中的成長係數、未來時間等參數從哪裡來。


## 1. 本階段確定使用的 Sensor Data

本階段只使用即時資料中確定會提供的欄位，避免使用 schema 有定義但實際資料不一定會進來的欄位。

| 物件 | 確定使用的 sensor | 中文名稱 |
|---|---|---|
| 濾網 Filter | `filter_diff_pressure_bar` | 濾網壓差 |
| 噴幅 Spray Width | `spray_width_mm` | 噴幅寬度 |
| 品質 Quality | `film_thickness_um` | 膜厚 |
| 噴嘴 Nozzle | `paint_flow_ml_min` | 塗料流量 |
| 空壓機 Air Compressor | `air_pressure_bar` | 空氣壓力 |
| 機械手臂 Robot Arm | `servo_torque_load_pct`, `path_error_mm` | 伺服負載、路徑誤差 |
| 環境 Environment | `temperature_c`, `humidity_rh` | 溫度、濕度 |

共同會用到的欄位：

| 欄位 | 中文名稱 | 用途 |
|---|---|---|
| `ts` | 時間戳記 | 計算 past / now / future 與變化率 |
| `station_id` | 站點 ID | 每個站點分開計算 |
| `data_quality_flag` | 資料品質標記 | 過濾異常或無效資料 |

## 2. Past / Now / Future 的時間定義

因為主要感測資料為 `sensor_1min`，也就是每 1 分鐘一筆資料，因此本文件時間單位統一使用 **min（分鐘）**。

| 名稱 | 建議值 | 中文說明 |
|---|---:|---|
| `recent_window_min` | 5 min | 計算目前值時，取最近 5 分鐘平均，避免單筆抖動 |
| `time_window_min` | 30 min | 計算變化率或成長係數時，取現在與 30 分鐘前比較 |
| `past_window_min` | 60 min | Past 模式使用過去 60 分鐘平均 |
| `future_window_min` | 30 min | Future 模式預設推估未來 30 分鐘 |

### 目前值 Now

$$
目前值 = 最近5分鐘資料平均
$$

例如：

$$
目前濾網壓差 = 最近5分鐘濾網壓差平均
$$

### 過去值 Past for Trend

用來計算變化率的過去值，建議取 30 分鐘前附近的 5 分鐘平均：

$$
過去值 = 30分鐘前附近5分鐘資料平均
$$

### 未來時間 Future Time

$$
未來時間 = future\_window\_min
$$

目前展示版先設定：

$$
未來時間 = 30\;min
$$

若未來 UI 或 API 有傳入查詢時間，例如未來 60 分鐘，則改用 UI / API 傳入的 `future_window_min`。

## 3. 線性公式與非線性公式的共同概念

### 3.1 線性變化率：一階差分法

本系統的資料是離散時間資料，所以線性變化率採用一階差分法：

$$
變化率 = \frac{目前值 - 過去值}{時間窗長度}
$$

用 sensor 欄位表示：

$$
X變化率 = \frac{X_{now} - X_{past}}{time\_window\_min}
$$

這個公式的意思是：在最近一段時間內，用線性斜率近似該 sensor 的變化趨勢。

### 3.2 非線性模型：指數型成長或衰減

部分物件不一定是線性變化。例如濾網阻塞可能前期慢、後期變快；噴嘴流量可能逐漸下降，也可能突然下降。因此可補充非線性模型。

若某個值會隨時間增加，可用指數成長模型：

$$
未來值 = 目前值 \times e^{成長係數 \times 未來時間}
$$

其中成長係數可由目前值與過去值反推：

$$
成長係數 = \frac{\ln(目前值 / 過去值)}{時間窗長度}
$$

若某個值會隨時間下降，可用指數衰減模型：

$$
未來值 = 目前值 \times e^{-衰減係數 \times 未來時間}
$$

其中衰減係數可由過去值與目前值反推：

$$
衰減係數 = \frac{\ln(過去值 / 目前值)}{時間窗長度}
$$

> 注意：非線性公式是補充模型，不代表每次都一定要使用。若資料量不足或變化不明顯，第一階段仍以線性公式為主。

# 4. 濾網 Filter 計算公式

## 4.1 使用資料

| sensor | 中文名稱 | 單位 |
|---|---|---|
| `filter_diff_pressure_bar` | 濾網壓差 | bar |
| `ts` | 時間戳記 | min |

濾網阻塞時，濾網前後壓差通常會逐漸上升，因此本階段以 `filter_diff_pressure_bar` 作為主要判斷資料。

## 4.2 濾網壓差目前值

$$
目前濾網壓差 = 最近5分鐘濾網壓差平均
$$

英文欄位表示：

$$
filter\_diff\_pressure\_{now} = mean(filter\_diff\_pressure\_bar\;in\;recent\;5\;min)
$$

## 4.3 濾網阻塞程度

以壓差換算阻塞程度：

$$
濾網阻塞程度(\%) = \frac{目前濾網壓差 - 正常壓差基準}{故障壓差門檻 - 正常壓差基準} \times 100\%
$$

本階段可先設定：

| 參數 | 中文說明 | 建議值 |
|---|---|---:|
| `正常壓差基準` | 濾網健康時的低壓差 | 0.1 bar |
| `故障壓差門檻` | 濾網判定為 fault 的壓差 | 0.7 bar |

因此：

$$
濾網阻塞程度(\%) = \frac{目前濾網壓差 - 0.1}{0.7 - 0.1} \times 100\%
$$

計算後需限制在 0% 到 100%。

## 4.4 線性公式：濾網壓差增加率

$$
濾網壓差增加率 = \frac{目前濾網壓差 - 過去濾網壓差}{時間窗長度}
$$

若時間窗長度設定為 30 分鐘：

$$
濾網壓差增加率 = \frac{目前濾網壓差 - 30分鐘前濾網壓差}{30}
$$

單位：

$$
bar/min
$$

這個公式是由斜率公式推導而來，也就是一階差分法。它表示濾網壓差每分鐘增加多少。

## 4.5 濾網剩餘可用時間 RUL

若濾網壓差正在上升，可以估算到達故障門檻還需要多久：

$$
濾網剩餘可用時間 = \frac{故障壓差門檻 - 目前濾網壓差}{濾網壓差增加率}
$$

代入門檻值：

$$
濾網剩餘可用時間 = \frac{0.7 - 目前濾網壓差}{濾網壓差增加率}
$$

單位：

$$
min
$$

若濾網壓差增加率小於等於 0，代表目前沒有惡化趨勢，RUL 可顯示為 stable 或暫不估算。

## 4.6 非線性公式：濾網壓差指數成長

濾網阻塞不一定是線性，可能前期變化慢，後期接近堵塞時壓差快速上升。因此可補充指數型模型：

$$
未來濾網壓差 = 目前濾網壓差 \times e^{阻塞成長係數 \times 未來時間}
$$

其中：

$$
阻塞成長係數 = \frac{\ln(目前濾網壓差 / 過去濾網壓差)}{時間窗長度}
$$

若使用 30 分鐘時間窗：

$$
阻塞成長係數 = \frac{\ln(目前濾網壓差 / 30分鐘前濾網壓差)}{30}
$$

| 參數 | 來源 |
|---|---|
| 目前濾網壓差 | 最近 5 分鐘 `filter_diff_pressure_bar` 平均 |
| 過去濾網壓差 | 30 分鐘前附近 5 分鐘 `filter_diff_pressure_bar` 平均 |
| 時間窗長度 | service 端設定，預設 30 min |
| 未來時間 | service 端預設 30 min，或由 UI / API 傳入 |

若目前濾網壓差沒有大於過去濾網壓差，代表沒有明顯阻塞成長，此非線性模型可暫不使用。

# 5. 噴幅 Spray Width 計算公式

## 5.1 使用資料

| sensor | 中文名稱 | 單位 |
|---|---|---|
| `spray_width_mm` | 噴幅寬度 | mm |
| `station_id` | 站點 ID | - |
| `ts` | 時間戳記 | min |

此版本資料已提供實際噴幅 `spray_width_mm`，因此優先用實測噴幅計算，不需要先用流量或壓力估測。

## 5.2 噴幅目前值

$$
目前噴幅 = 最近5分鐘噴幅平均
$$

英文欄位表示：

$$
spray\_width\_{now} = mean(spray\_width\_mm\;in\;recent\;5\;min)
$$

## 5.3 噴幅縮減率

由於不同站點的噴幅基準不同，因此建議用各站自己的健康基準計算縮減率。

$$
噴幅縮減率(\%) = \frac{站點噴幅基準 - 目前噴幅}{站點噴幅基準} \times 100\%
$$

其中：

$$
站點噴幅基準 = 該站點過去健康資料的噴幅中位數
$$

若噴幅縮減率為正，代表目前噴幅比健康基準變窄。

## 5.4 線性公式：噴幅變化率

$$
噴幅變化率 = \frac{目前噴幅 - 過去噴幅}{時間窗長度}
$$

若使用 30 分鐘時間窗：

$$
噴幅變化率 = \frac{目前噴幅 - 30分鐘前噴幅}{30}
$$

單位：

$$
mm/min
$$

若噴幅變化率小於 0，代表噴幅正在變窄。

## 5.5 非線性公式：噴幅指數縮減

噴幅變窄不一定是線性，也可能前期幾乎不變，阻塞或磨耗到某程度後快速變窄。因此可補充指數衰減模型：

$$
未來噴幅 = 目前噴幅 \times e^{-噴幅縮減係數 \times 未來時間}
$$

其中：

$$
噴幅縮減係數 = \frac{\ln(過去噴幅 / 目前噴幅)}{時間窗長度}
$$

若使用 30 分鐘時間窗：

$$
噴幅縮減係數 = \frac{\ln(30分鐘前噴幅 / 目前噴幅)}{30}
$$

| 參數 | 來源 |
|---|---|
| 目前噴幅 | 最近 5 分鐘 `spray_width_mm` 平均 |
| 過去噴幅 | 30 分鐘前附近 5 分鐘 `spray_width_mm` 平均 |
| 時間窗長度 | service 端設定，預設 30 min |
| 未來時間 | service 端預設 30 min，或由 UI / API 傳入 |

若目前噴幅沒有小於過去噴幅，代表沒有明顯縮減趨勢，此非線性模型可暫不使用。

# 6. 品質模組 Quality 計算公式

## 6.1 使用資料

| sensor | 中文名稱 | 單位 |
|---|---|---|
| `film_thickness_um` | 膜厚 | um |
| `ts` | 時間戳記 | min |

品質模組主要使用膜厚判斷塗層品質是否偏離目標值。

## 6.2 膜厚目前值與誤差

$$
目前膜厚 = 最近5分鐘膜厚平均
$$

目標膜厚先設定為：

$$
目標膜厚 = 15.0\;um
$$

膜厚誤差：

$$
膜厚誤差 = 目前膜厚 - 目標膜厚
$$

膜厚誤差百分比：

$$
膜厚誤差百分比 = \frac{目前膜厚 - 目標膜厚}{目標膜厚} \times 100\%
$$

## 6.3 品質分數

可用膜厚偏離程度估算品質分數。品質分數的目的不是取代 threshold rule，而是把膜厚偏差轉成 UI 比較容易理解的百分比分數。

$$
品質分數 = 100 - |目前膜厚 - 目標膜厚| \times 品質扣分係數
$$

本階段設定：

$$
品質扣分係數 = 40\;分/\mu m
$$

因此公式可寫成：

$$
品質分數 = 100 - |目前膜厚 - 目標膜厚| \times 40
$$

其中 `40` 而是設計的扣分係數。它代表膜厚每偏離目標值 `1 μm`，品質分數扣 `40 分`。

### 為什麼設定為 40？

目前膜厚目標值為：

$$
目標膜厚 = 15.0\;\mu m
$$

膜厚判斷門檻為：

| 膜厚範圍 | 狀態 | 與目標值的偏差 | 品質分數概念 |
|---|---|---:|---:|
| 14.5 ~ 15.5 μm | normal | 約 0.5 μm 以內 | 約 80 分以上 |
| 14.0 ~ 14.5 μm 或 15.5 ~ 16.0 μm | warning | 約 0.5 ~ 1.0 μm | 約 60 ~ 80 分 |
| 小於 14.0 μm 或大於 16.0 μm | fault | 超過 1.0 μm | 約 60 分以下 |

如果希望膜厚完全符合目標時為 `100 分`，偏差 `1 μm` 時約下降到 `60 分`，則扣分係數可以由下式得到：

$$
品質扣分係數 = \frac{100 - 60}{1\;\mu m} = 40\;分/\mu m
$$

所以 `×40` 的用途是讓品質分數能大致對應 normal / warning / fault 的膜厚門檻。

### 範例

若目前膜厚為 `15.3 μm`：

$$
品質分數 = 100 - |15.3 - 15.0| \times 40 = 88
$$

若目前膜厚為 `16.0 μm`：

$$
品質分數 = 100 - |16.0 - 15.0| \times 40 = 60
$$

計算後限制在 0 到 100 之間：

$$
0 \le 品質分數 \le 100
$$

此公式代表膜厚越接近目標值，品質分數越高；膜厚偏離越大，品質分數越低。



## 6.4 線性公式：膜厚變化率

$$
膜厚變化率 = \frac{目前膜厚 - 過去膜厚}{時間窗長度}
$$

若使用 30 分鐘時間窗：

$$
膜厚變化率 = \frac{目前膜厚 - 30分鐘前膜厚}{30}
$$

單位：

$$
um/min
$$

此公式用來觀察膜厚是否逐漸偏厚或偏薄。

## 6.5 非線性公式：膜厚誤差擴大模型

膜厚本身不一定會單方向增加或下降，因此非線性模型不直接預測膜厚，而是預測「膜厚誤差」是否擴大。

定義：

$$
目前膜厚誤差量 = |目前膜厚 - 目標膜厚|
$$

$$
過去膜厚誤差量 = |過去膜厚 - 目標膜厚|
$$

非線性模型：

$$
未來膜厚誤差量 = 目前膜厚誤差量 \times e^{誤差成長係數 \times 未來時間}
$$

其中：

$$
誤差成長係數 = \frac{\ln(目前膜厚誤差量 / 過去膜厚誤差量)}{時間窗長度}
$$

| 參數 | 來源 |
|---|---|
| 目前膜厚誤差量 | 最近 5 分鐘膜厚平均與目標膜厚的差 |
| 過去膜厚誤差量 | 30 分鐘前膜厚平均與目標膜厚的差 |
| 時間窗長度 | service 端設定，預設 30 min |
| 未來時間 | service 端預設 30 min，或由 UI / API 傳入 |

若目前膜厚誤差沒有大於過去膜厚誤差，代表品質偏差沒有擴大，此模型可暫不使用。

# 7. 噴嘴 Nozzle 計算公式

## 7.1 使用資料

| sensor | 中文名稱 | 單位 |
|---|---|---|
| `paint_flow_ml_min` | 塗料流量 | ml/min |
| `ts` | 時間戳記 | min |

噴嘴阻塞或供料不足時，塗料流量通常會下降，因此以 `paint_flow_ml_min` 作為主要計算資料。

## 7.2 塗料流量目前值與偏差

$$
目前塗料流量 = 最近5分鐘塗料流量平均
$$

正常流量範圍可先使用 105 到 125 ml/min，因此取中間值：

$$
正常塗料流量 = 115\;ml/min
$$

流量偏差：

$$
流量偏差 = 目前塗料流量 - 正常塗料流量
$$

流量偏差百分比：

$$
流量偏差百分比 = \frac{目前塗料流量 - 正常塗料流量}{正常塗料流量} \times 100\%
$$

## 7.3 噴嘴阻塞指標

當流量低於正常值時，可用流量下降程度估算噴嘴阻塞指標：

$$
噴嘴阻塞指標(\%) = \frac{正常塗料流量 - 目前塗料流量}{正常塗料流量 - 故障流量下限} \times 100\%
$$

本階段可設定：

| 參數 | 中文說明 | 建議值 |
|---|---|---:|
| 正常塗料流量 | 正常流量中心值 | 115 ml/min |
| 故障流量下限 | 低於此值視為 fault | 95 ml/min |

因此：

$$
噴嘴阻塞指標(\%) = \frac{115 - 目前塗料流量}{115 - 95} \times 100\%
$$

計算後限制在 0% 到 100%。

## 7.4 線性公式：流量變化率

$$
流量變化率 = \frac{目前塗料流量 - 過去塗料流量}{時間窗長度}
$$

若使用 30 分鐘時間窗：

$$
流量變化率 = \frac{目前塗料流量 - 30分鐘前塗料流量}{30}
$$

單位：

$$
(ml/min)/min
$$

若流量變化率小於 0，代表塗料流量正在下降。

## 7.5 非線性公式：塗料流量指數衰減

噴嘴堵塞可能不是線性下降，可能在某段時間後流量快速下降，因此可補充指數衰減模型：

$$
未來塗料流量 = 目前塗料流量 \times e^{-流量衰減係數 \times 未來時間}
$$

其中：

$$
流量衰減係數 = \frac{\ln(過去塗料流量 / 目前塗料流量)}{時間窗長度}
$$

若使用 30 分鐘時間窗：

$$
流量衰減係數 = \frac{\ln(30分鐘前塗料流量 / 目前塗料流量)}{30}
$$

| 參數 | 來源 |
|---|---|
| 目前塗料流量 | 最近 5 分鐘 `paint_flow_ml_min` 平均 |
| 過去塗料流量 | 30 分鐘前附近 5 分鐘 `paint_flow_ml_min` 平均 |
| 時間窗長度 | service 端設定，預設 30 min |
| 未來時間 | service 端預設 30 min，或由 UI / API 傳入 |

若目前流量沒有小於過去流量，代表沒有明顯流量衰減，此非線性模型可暫不使用。

# 8. 空壓機 Air Compressor 計算公式

## 8.1 使用資料

| sensor | 中文名稱 | 單位 |
|---|---|---|
| `air_pressure_bar` | 空氣壓力 | bar |
| `ts` | 時間戳記 | min |

空壓機不一定是線性損耗，常見問題可能是壓力偏低、偏高或波動過大。因此除了平均壓力，也要看壓力穩定度。

## 8.2 空壓目前值與偏差

$$
目前空壓 = 最近5分鐘空氣壓力平均
$$

正常範圍可先設定為 2.7 到 3.8 bar，因此中心值為：

$$
正常空壓 = \frac{2.7 + 3.8}{2} = 3.25\;bar
$$

空壓偏差：

$$
空壓偏差 = 目前空壓 - 正常空壓
$$

空壓偏差百分比：

$$
空壓偏差百分比 = \frac{目前空壓 - 正常空壓}{正常空壓} \times 100\%
$$

## 8.3 線性公式：空壓變化率

$$
空壓變化率 = \frac{目前空壓 - 過去空壓}{時間窗長度}
$$

若使用 30 分鐘時間窗：

$$
空壓變化率 = \frac{目前空壓 - 30分鐘前空壓}{30}
$$

單位：

$$
bar/min
$$

此公式可用來觀察壓力是否逐漸上升或下降。

## 8.4 非線性補充：空壓波動指標

空壓機的異常不一定是指數成長或衰減，比較常見的是壓力波動，因此非線性補充建議使用平方偏差或 RMS 指標。

空壓波動指標：

$$
空壓波動指標 = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(第i筆空壓 - 正常空壓)^2}
$$

其中：

| 參數 | 來源 |
|---|---|
| 第 i 筆空壓 | 時間窗內每一筆 `air_pressure_bar` |
| 正常空壓 | 正常範圍中心值，預設 3.25 bar |
| n | 時間窗內資料筆數，例如 30 分鐘約 30 筆 |
| 時間窗長度 | service 端設定，預設 30 min |

這個公式不是用未來時間做預測，而是用來判斷某段時間內空壓是否穩定。若需要做 future 推估，可另外使用線性空壓變化率進行短期預測。

# 9. 機械手臂 Robot Arm 計算公式

## 9.1 使用資料

| sensor | 中文名稱 | 單位 |
|---|---|---|
| `servo_torque_load_pct` | 伺服扭矩負載 | % |
| `path_error_mm` | 路徑誤差 | mm |
| `ts` | 時間戳記 | min |

機械手臂狀態以伺服負載與路徑誤差共同判斷。任一項明顯異常都可能代表機構或控制狀態有問題。

## 9.2 伺服負載指標

$$
目前伺服負載 = 最近5分鐘伺服負載平均
$$

可用 40% 作為健康基準，75% 作為故障門檻：

$$
伺服負載指標(\%) = \frac{目前伺服負載 - 40}{75 - 40} \times 100\%
$$

計算後限制在 0% 到 100%。

## 9.3 路徑誤差指標

$$
目前路徑誤差 = 最近5分鐘路徑誤差平均
$$

故障門檻可先設定為 0.15 mm：

$$
路徑誤差指標(\%) = \frac{目前路徑誤差}{0.15} \times 100\%
$$

計算後限制在 0% 到 100%。

## 9.4 線性公式：伺服負載與路徑誤差變化率

伺服負載變化率：

$$
伺服負載變化率 = \frac{目前伺服負載 - 過去伺服負載}{時間窗長度}
$$

路徑誤差變化率：

$$
路徑誤差變化率 = \frac{目前路徑誤差 - 過去路徑誤差}{時間窗長度}
$$

若使用 30 分鐘時間窗：

$$
伺服負載變化率 = \frac{目前伺服負載 - 30分鐘前伺服負載}{30}
$$

$$
路徑誤差變化率 = \frac{目前路徑誤差 - 30分鐘前路徑誤差}{30}
$$

## 9.5 非線性補充：平方和綜合風險

機械手臂不一定是單一 sensor 線性變差，有可能是伺服負載與路徑誤差同時變大。因此可用平方和方式放大嚴重偏差。

$$
機械手臂綜合風險 = \sqrt{\frac{伺服負載指標^2 + 路徑誤差指標^2}{2}}
$$

| 參數 | 來源 |
|---|---|
| 伺服負載指標 | 由 `servo_torque_load_pct` 正規化取得 |
| 路徑誤差指標 | 由 `path_error_mm` 正規化取得 |
| 2 | 代表目前使用兩個主要指標 |

此公式不需要未來時間，主要用於目前狀態的非線性風險合成。若要做 future 推估，可分別使用伺服負載變化率與路徑誤差變化率往未來推估。

# 10. 環境 Environment 計算公式

## 10.1 使用資料

| sensor | 中文名稱 | 單位 |
|---|---|---|
| `temperature_c` | 環境溫度 | °C |
| `humidity_rh` | 環境濕度 | %RH |
| `ts` | 時間戳記 | min |

環境資料主要作為品質異常的輔助判斷。

## 10.2 溫度與濕度目前值

$$
目前溫度 = 最近5分鐘或最近有效環境資料平均
$$

$$
目前濕度 = 最近5分鐘或最近有效環境資料平均
$$

若環境資料來自 `sensor_3min`，則最近 5 分鐘可能只有 1 到 2 筆資料，可改用最近有效值或最近 15 分鐘平均。

## 10.3 線性公式：溫度與濕度變化率

溫度變化率：

$$
溫度變化率 = \frac{目前溫度 - 過去溫度}{時間窗長度}
$$

濕度變化率：

$$
濕度變化率 = \frac{目前濕度 - 過去濕度}{時間窗長度}
$$

單位分別為：

$$
^\circ C/min
$$

$$
\%RH/min
$$

## 10.4 非線性補充：溫濕度偏離指標

環境異常不一定是線性增加或下降，比較適合看目前溫濕度與正常中心值的偏離程度。

可先設定：

$$
正常溫度 = 25^\circ C
$$

$$
正常濕度 = 55\%RH
$$

定義偏離指標：

$$
環境偏離指標 = \sqrt{\left(\frac{目前溫度 - 正常溫度}{溫度容許範圍}\right)^2 + \left(\frac{目前濕度 - 正常濕度}{濕度容許範圍}\right)^2}
$$

其中：

| 參數 | 來源 |
|---|---|
| 正常溫度 | 正常範圍 20 至 30°C 的中心值 |
| 正常濕度 | 正常範圍 45 至 65%RH 的中心值 |
| 溫度容許範圍 | 可取 5°C，也就是 20 至 30°C 的半寬 |
| 濕度容許範圍 | 可取 10%RH，也就是 45 至 65%RH 的半寬 |

此公式不需要未來時間，主要用於判斷目前環境是否偏離正常條件。

# 11. 狀態判斷方式整理

各物件最後都會回到 normal / warning / fault 判斷。第一階段建議以 threshold rule 為主，線性與非線性公式作為趨勢或風險補充。

| 物件 | 主要判斷值 | 線性補充 | 非線性補充 |
|---|---|---|---|
| 濾網 | 濾網壓差、阻塞程度 | 壓差增加率、RUL | 指數型壓差成長 |
| 噴幅 | 噴幅寬度、縮減率 | 噴幅變化率 | 指數型噴幅縮減 |
| 品質 | 膜厚誤差、品質分數 | 膜厚變化率 | 膜厚誤差擴大模型 |
| 噴嘴 | 塗料流量、阻塞指標 | 流量變化率 | 流量指數衰減 |
| 空壓機 | 空壓平均值 | 空壓變化率 | 空壓 RMS 波動指標 |
| 機械手臂 | 伺服負載、路徑誤差 | 負載與誤差變化率 | 平方和綜合風險 |
| 環境 | 溫度、濕度 | 溫濕度變化率 | 溫濕度偏離指標 |